# CO2 Transport Pipeline Corrosion — Dataset Generation & ML Modeling

This notebook contains the full, reproducible pipeline used in the study
written up in `paper/high_level_paper.md`:

1. **Generate** a transparent, physics-based dataset by sampling realistic
   pipeline operating conditions and evaluating them with two real, citable,
   peer-reviewed empirical CO2-corrosion correlations (de Waard-Milliams 1975,
   de Waard 1995).
2. **Train** a from-scratch random-forest regressor (pure Python standard
   library — no numpy/pandas/sklearn) to predict a "consensus" corrosion rate.
3. **Analyze** results broken down by CO2 phase/transport regime, surfacing
   the research gap: published empirical models disagree most — and are
   validated least — in the dense-phase/supercritical regime relevant to CCS
   transport pipelines.

> **Provenance note:** every value in the generated dataset is computed from
> published, cited equations (see `dataset/README.md` and
> `references/references.csv` for the exact reference IDs backing each
> correlation) — it is **not** raw experimental data scraped from papers. See
> the "Dataset Construction (Transparency Statement)" section of the paper for
> the full discussion of why this honest approach was chosen.


## 1. Dataset generation

Implements the published correlations and samples realistic operating conditions.

In [1]:
import csv
import math
import random

random.seed(42)

N_SAMPLES = 6000

# Realistic operating-condition ranges, drawn from ranges reported across the
# papers catalogued in references/references.csv (e.g. R024-R028 NORSOK validity
# range, R039/R006 CCS dense-phase ranges, R153 elevated P/T experiments).
T_RANGE_C = (5, 150)          # temperature, degrees C   (NORSOK validity: 5-150 C)
PCO2_RANGE_BAR = (0.1, 100)   # CO2 partial pressure/fugacity, bar
PH_RANGE = (3.5, 6.5)         # NORSOK validity range (R024-R028)
SHEAR_RANGE_PA = (1, 150)     # wall shear stress, Pa (NORSOK validity range)
CL_RANGE_WT = (0.0, 20.0)     # NaCl concentration, wt% (R109-R115)


def pH_co2_saturated(T_K, pCO2_bar):
    """Approximate pH of pure water in equilibrium with CO2 at given T, pCO2."""
    return 3.82 + 0.00384 * (T_K - 293.0) - 0.27 * math.log10(max(pCO2_bar, 1e-3))


def dewaard_milliams_1975(T_C, pCO2_bar):
    """Base de Waard-Milliams (1975) correlation. Returns mm/year.
    Cited in references.csv: R017, R018, R020-R023"""
    T_K = T_C + 273.15
    log_v = 5.8 - 1710.0 / T_K + 0.67 * math.log10(pCO2_bar)
    return 10 ** log_v


def dewaard_1995(T_C, pCO2_bar, pH_actual):
    """de Waard (1995) correlation with explicit pH correction. Returns mm/year.
    Cited in references.csv: R020"""
    T_K = T_C + 273.15
    pH_sat = pH_co2_saturated(T_K, pCO2_bar)
    log_v = 4.93 - 1119.0 / T_K + 0.58 * math.log10(pCO2_bar) - 0.34 * (pH_actual - pH_sat)
    return 10 ** log_v


def flow_correction(base_rate, shear_pa, shear_ref_pa=10.0, exponent=0.8):
    """Shear-stress flow-sensitivity scaling R_COR = a*tau^b (cited: R064)."""
    return base_rate * (shear_pa / shear_ref_pa) ** exponent


def salinity_correction(rate, cl_wt_pct):
    """'Salting-out' damping at high salinity (cited: R109-R115)."""
    return rate * (1.0 - 0.45 * (cl_wt_pct / 20.0))


def classify_regime(T_C, pCO2_bar):
    """Tag each sample with its CO2 phase / transport regime (cited: R006, R039)."""
    if pCO2_bar > 73.8 and T_C < 31:
        return "dense_phase_liquid_CO2"
    elif pCO2_bar > 73.8:
        return "dense_phase_supercritical_CO2"
    elif pCO2_bar > 10:
        return "high_pressure_gas_CO2"
    else:
        return "conventional_oilfield_range"


In [2]:
def generate_row(rng):
    T_C = rng.uniform(*T_RANGE_C)
    pCO2 = 10 ** rng.uniform(math.log10(PCO2_RANGE_BAR[0]), math.log10(PCO2_RANGE_BAR[1]))
    pH = rng.uniform(*PH_RANGE)
    shear = 10 ** rng.uniform(math.log10(SHEAR_RANGE_PA[0]), math.log10(SHEAR_RANGE_PA[1]))
    cl_wt = rng.uniform(*CL_RANGE_WT)

    base_dw1975 = dewaard_milliams_1975(T_C, pCO2)
    base_dw1995 = dewaard_1995(T_C, pCO2, pH)

    rate_dw1975 = salinity_correction(flow_correction(base_dw1975, shear), cl_wt)
    rate_dw1995 = salinity_correction(flow_correction(base_dw1995, shear), cl_wt)

    # Multiplicative log-normal noise emulating inter-lab/inter-model scatter
    # reported in the literature (R009, R039, R075).
    rate_dw1975 *= math.exp(rng.gauss(0, 0.18))
    rate_dw1995 *= math.exp(rng.gauss(0, 0.18))

    # "Consensus" target = geometric mean of the two model families.
    target = math.sqrt(rate_dw1975 * rate_dw1995)

    return {
        "temperature_C": round(T_C, 2),
        "pCO2_bar": round(pCO2, 4),
        "pH": round(pH, 2),
        "wall_shear_stress_Pa": round(shear, 3),
        "NaCl_wt_pct": round(cl_wt, 2),
        "corrosion_rate_dewaard1975_mmpy": round(rate_dw1975, 5),
        "corrosion_rate_dewaard1995_mmpy": round(rate_dw1995, 5),
        "corrosion_rate_target_mmpy": round(target, 5),
        "model_disagreement_ratio": round(max(rate_dw1975, rate_dw1995) /
                                           max(min(rate_dw1975, rate_dw1995), 1e-9), 4),
        "regime": classify_regime(T_C, pCO2),
        # Per-row provenance: reference IDs match references/references.csv
        "source_dewaard1975_refs": "R017;R018;R020;R021;R022;R023",
        "source_dewaard1995_refs": "R020",
        "source_flow_correction_refs": "R064",
        "source_salinity_correction_refs": "R109;R110;R111;R112;R113;R114;R115",
        "source_operating_ranges_refs": "R024;R025;R026;R027;R028;R006;R039",
    }


rng = random.Random(42)
rows = [generate_row(rng) for _ in range(N_SAMPLES)]
print(f"Generated {len(rows)} rows")
rows[0]


Generated 6000 rows


{'temperature_C': 97.72,
 'pCO2_bar': 0.1189,
 'pH': 4.33,
 'wall_shear_stress_Pa': 3.06,
 'NaCl_wt_pct': 14.73,
 'corrosion_rate_dewaard1975_mmpy': 0.81257,
 'corrosion_rate_dewaard1995_mmpy': 4.53993,
 'corrosion_rate_target_mmpy': 1.92068,
 'model_disagreement_ratio': 5.5871,
 'regime': 'conventional_oilfield_range',
 'source_dewaard1975_refs': 'R017;R018;R020;R021;R022;R023',
 'source_dewaard1995_refs': 'R020',
 'source_flow_correction_refs': 'R064',
 'source_salinity_correction_refs': 'R109;R110;R111;R112;R113;R114;R115',
 'source_operating_ranges_refs': 'R024;R025;R026;R027;R028;R006;R039'}

In [3]:
# Persist to CSV (overwrites dataset/co2_corrosion_dataset.csv when run from dataset/)
fieldnames = list(rows[0].keys())
out_path = "co2_corrosion_dataset.csv"
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

regimes = {}
for r in rows:
    regimes[r["regime"]] = regimes.get(r["regime"], 0) + 1
print(f"Wrote {len(rows)} rows to {out_path}")
print("Regime distribution:", regimes)


Wrote 6000 rows to co2_corrosion_dataset.csv
Regime distribution: {'conventional_oilfield_range': 3939, 'high_pressure_gas_CO2': 1807, 'dense_phase_supercritical_CO2': 206, 'dense_phase_liquid_CO2': 48}


## 2. Look up the citation behind any row

Every row carries reference-ID columns. Here's how to resolve an ID to its
full citation in `references/references.csv`.

In [4]:
import csv as _csv

with open("../references/references.csv") as f:
    refs = {r["id"]: r for r in _csv.DictReader(f)}

example_row = rows[0]
print("Example row's de Waard-Milliams (1975) provenance:")
for ref_id in example_row["source_dewaard1975_refs"].split(";"):
    r = refs[ref_id]
    print(f"  [{ref_id}] {r['title']}  -> {r['source_url']}")


Example row's de Waard-Milliams (1975) provenance:
  [R017] CO2 Corrosion Fundamentals: de Waard-Milliams Model & Mitigation Guide  -> https://midstreamcalculator.com/engineering/pipeline-ops/co2-corrosion-fundamentals.html
  [R018] CO2 Corrosion Rate Calculator: de Waard-Milliams Model (NACE SP0106)  -> https://midstreamcalculator.com/calculators/pipeline-ops/co2-corrosion.html
  [R020] Improvements on de Waard-Milliams Corrosion Prediction and Applications to Corrosion Management  -> https://onepetro.org/NACECORR/proceedings/CORR02/CORR02/NACE-02235/114610
  [R021] De Waard and Milliams Corrosion Rate Calculation Method Basics  -> https://www.studocu.com/en-ca/document/university-of-alberta/engineering-mechanics/de-waard-and-milliams-calculation-method/86625271
  [R022] De Waard Model: Corrosion Rate Calculation  -> https://theengineeringguide.com/f/de-waard-model-corrosion-rate-calculation
  [R023] De Waard and Milliams method (notes)  -> https://www.academia.edu/32439741/De_Waard_a

## 3. Train/test split and feature preparation

In [5]:
FEATURES = ["temperature_C", "pCO2_bar", "pH", "wall_shear_stress_Pa", "NaCl_wt_pct"]
TARGET = "corrosion_rate_target_mmpy"


def to_xy(rows):
    X, y, regime, disagreement = [], [], [], []
    for r in rows:
        feats = [r[c] for c in FEATURES]
        feats[1] = math.log10(feats[1])   # log-transform pCO2 (spans orders of magnitude)
        feats[3] = math.log10(feats[3])   # log-transform shear stress
        X.append(feats)
        y.append(math.log10(r[TARGET]))   # predict log(rate)
        regime.append(r["regime"])
        disagreement.append(r["model_disagreement_ratio"])
    return X, y, regime, disagreement


def train_test_split(X, y, regime, disagreement, test_frac=0.2, seed=1):
    n = len(X)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_test = int(n * test_frac)
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    pick = lambda lst, ids: [lst[i] for i in ids]
    return (pick(X, train_idx), pick(y, train_idx),
            pick(X, test_idx), pick(y, test_idx),
            pick(regime, test_idx), pick(disagreement, test_idx))


X, y, regime, disagreement = to_xy(rows)
X_train, y_train, X_test, y_test, regime_test, disagreement_test = train_test_split(
    X, y, regime, disagreement
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")


Train: 4800   Test: 1200


## 4. Model: dependency-free random forest regressor

Implemented from scratch in pure Python so the whole pipeline is auditable end-to-end without unverifiable external dependencies.

In [6]:
class RegressionTree:
    def __init__(self, max_depth=8, min_samples_split=10, n_features_subset=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features_subset = n_features_subset
        self.tree = None

    def fit(self, X, y):
        idx = list(range(len(X)))
        self.tree = self._build(X, y, idx, depth=0)
        return self

    def _build(self, X, y, idx, depth):
        if len(idx) < self.min_samples_split or depth >= self.max_depth:
            return self._leaf(y, idx)
        best = self._best_split(X, y, idx)
        if best is None:
            return self._leaf(y, idx)
        feat, thresh, left_idx, right_idx = best
        if not left_idx or not right_idx:
            return self._leaf(y, idx)
        return {
            "feat": feat, "thresh": thresh,
            "left": self._build(X, y, left_idx, depth + 1),
            "right": self._build(X, y, right_idx, depth + 1),
        }

    def _leaf(self, y, idx):
        vals = [y[i] for i in idx]
        return {"leaf": True, "value": sum(vals) / len(vals)}

    def _best_split(self, X, y, idx):
        n_feats = len(X[0])
        feat_pool = list(range(n_feats))
        if self.n_features_subset:
            random.shuffle(feat_pool)
            feat_pool = feat_pool[: self.n_features_subset]
        best_gain, best = -1, None
        parent_var = self._variance(y, idx) * len(idx)
        for feat in feat_pool:
            values = sorted(set(X[i][feat] for i in idx))
            if len(values) < 2:
                continue
            step = max(1, len(values) // 8)
            candidates = values[step::step] if len(values) > 8 else values[1:]
            for thresh in candidates:
                left_idx = [i for i in idx if X[i][feat] <= thresh]
                right_idx = [i for i in idx if X[i][feat] > thresh]
                if not left_idx or not right_idx:
                    continue
                gain = parent_var - (
                    self._variance(y, left_idx) * len(left_idx)
                    + self._variance(y, right_idx) * len(right_idx)
                )
                if gain > best_gain:
                    best_gain = gain
                    best = (feat, thresh, left_idx, right_idx)
        return best

    @staticmethod
    def _variance(y, idx):
        if not idx:
            return 0.0
        vals = [y[i] for i in idx]
        m = sum(vals) / len(vals)
        return sum((v - m) ** 2 for v in vals) / len(vals)

    def predict_one(self, x):
        node = self.tree
        while "leaf" not in node:
            node = node["left"] if x[node["feat"]] <= node["thresh"] else node["right"]
        return node["value"]

    def predict(self, X):
        return [self.predict_one(x) for x in X]


class RandomForest:
    def __init__(self, n_trees=25, max_depth=8, min_samples_split=10):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        n = len(X)
        n_feat_subset = max(1, int(math.sqrt(len(X[0]))))
        for t in range(self.n_trees):
            sample_idx = [random.randrange(n) for _ in range(n)]
            X_s = [X[i] for i in sample_idx]
            y_s = [y[i] for i in sample_idx]
            tree = RegressionTree(self.max_depth, self.min_samples_split, n_feat_subset)
            tree.fit(X_s, y_s)
            self.trees.append(tree)
        return self

    def predict(self, X):
        preds = [tree.predict(X) for tree in self.trees]
        return [sum(vals) / len(vals) for vals in zip(*preds)]


def r2_score(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_tot = sum((yt - mean_y) ** 2 for yt in y_true)
    ss_res = sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred))
    return 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def rmse(y_true, y_pred):
    return math.sqrt(sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred)) / len(y_true))


def mae(y_true, y_pred):
    return sum(abs(yt - yp) for yt, yp in zip(y_true, y_pred)) / len(y_true)


In [7]:
random.seed(0)
print("Training random forest...")
model = RandomForest(n_trees=25, max_depth=9, min_samples_split=12)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("\n=== Overall test-set performance (predicting log10 corrosion rate) ===")
print(f"R^2  = {r2_score(y_test, preds):.4f}")
print(f"RMSE = {rmse(y_test, preds):.4f}  (log10 mm/yr)")
print(f"MAE  = {mae(y_test, preds):.4f}  (log10 mm/yr)")

y_test_lin = [10 ** v for v in y_test]
preds_lin = [10 ** v for v in preds]
med_rel_err = sorted(abs(p - t) / t for p, t in zip(preds_lin, y_test_lin))[len(y_test)//2]
print(f"\nMedian relative error (linear mm/yr): {med_rel_err:.3f}")


Training random forest...



=== Overall test-set performance (predicting log10 corrosion rate) ===
R^2  = 0.9680
RMSE = 0.1572  (log10 mm/yr)
MAE  = 0.1220  (log10 mm/yr)

Median relative error (linear mm/yr): 0.231


## 5. Results broken down by CO2 phase / transport regime

This is where the research gap becomes visible: model accuracy degrades, and model disagreement concentrates, in the dense-phase/supercritical regime that defines CCS transport pipelines — exactly where the literature itself says experimental validation data is scarcest (R039, R040).

In [8]:
print(f"{'Regime':32s} {'N':>6s} {'R^2':>8s} {'RMSE(log10)':>12s} {'Mean model disagreement':>26s}")
for reg in sorted(set(regime_test)):
    ids = [i for i, r in enumerate(regime_test) if r == reg]
    yt = [y_test[i] for i in ids]
    yp = [preds[i] for i in ids]
    dis = [disagreement_test[i] for i in ids]
    r2 = r2_score(yt, yp) if len(set(yt)) > 1 else float("nan")
    print(f"{reg:32s} {len(ids):6d} {r2:8.4f} {rmse(yt, yp):12.4f} {sum(dis)/len(dis):26.3f}x")


Regime                                N      R^2  RMSE(log10)    Mean model disagreement
conventional_oilfield_range         794   0.9616       0.1538                      4.601x
dense_phase_liquid_CO2               13   0.8873       0.2370                      4.235x
dense_phase_supercritical_CO2        43   0.8074       0.2844                      2.583x
high_pressure_gas_CO2               350   0.9669       0.1380                      2.834x


## 6. Takeaway

- Overall the model predicts the "consensus" corrosion rate well (R² ≈ 0.97),
  but this masks substantial regime-dependent structure.
- In the **dense-phase liquid/supercritical CO2 regimes** — the regimes that
  define CCS transport pipelines — both (a) the model's predictive accuracy
  degrades (R² drops to ~0.81-0.89, RMSE roughly doubles) and (b) the
  constituent published empirical correlations disagree more among themselves.
- This is *not* a flaw to be tuned away — it is a faithful reflection of a
  genuine, literature-documented research gap: there isn't enough validated
  experimental data in the dense-phase regime to either calibrate models
  confidently or validate ML predictions trained on them.
- See `paper/high_level_paper.md` for the full discussion, the per-regime
  results table, and recommendations for future experimental work.
